# Graph Walk — task demo

Three sections:

1. **Causal model** — graph construction, random walk, valid neighbors.
2. **Templates & token positions** — how the trailing-separator prompt is rendered and where `last` lands.
3. **Counterfactual generators** — coordinate cycling for balanced coverage.

Tokenization uses `gpt2`. We use a tiny `ring(6)` graph and a 32-step walk so the demo stays cheap. The full task uses `context_length=2048` on multi-thousand-node graphs.

In [1]:
from causalab.tasks.graph_walk import (
    GraphWalkConfig,
    build_graph,
    create_causal_model,
    create_token_positions,
)
from causalab.tasks.graph_walk.counterfactuals import generate_dataset

# Tiny ring graph + short walk — keeps the demo fast and the prompt readable.
cfg = GraphWalkConfig(
    graph_type="ring",
    graph_size=6,
    context_length=32,
    seed=0,
)
graph = build_graph(cfg.graph_type, cfg.graph_size, cfg.graph_size_2)
model = create_causal_model(cfg)
(model.id, graph.n_nodes, cfg.concepts, graph.adjacency)

('graph_walk',
 6,
 ['time', 'lot', 'hint', 'film', 'mind', 'den'],
 {0: [5, 1], 1: [0, 2], 2: [1, 3], 3: [2, 4], 4: [3, 5], 5: [4, 0]})

## 1. Causal model

DAG: `node_coordinates → walk_sequence → raw_input`, plus `node_coordinates → raw_output`. `raw_input` and `raw_output` are lazy — they only get computed when accessed.

Sample one input. The walk is generated forward from the target node and reversed, so it *ends* at `node_coordinates`.

In [2]:
trace = model.sample_input()

print(f"node_coordinates  = {trace['node_coordinates']}")
print(f"walk_sequence     = {trace['walk_sequence']}")
print(f"raw_output        = {trace['raw_output']}    (= valid neighbors of the final node)")
print()
print("raw_input:")
print(f"  {trace['raw_input']!r}")

node_coordinates  = (2.0943951023931953,)
walk_sequence     = [3, 2, 1, 0, 5, 4, 3, 2, 1, 0, 5, 4, 3, 2, 1, 0, 5, 4, 3, 2, 1, 0, 5, 4, 3, 2, 1, 0, 5, 4, 3, 2]
raw_output        = ['lot', 'film']    (= valid neighbors of the final node)

raw_input:
  'film|hint|lot|time|den|mind|film|hint|lot|time|den|mind|film|hint|lot|time|den|mind|film|hint|lot|time|den|mind|film|hint|lot|time|den|mind|film|hint|'


Note the trailing `|` — that's intentional. It places the model at the position it would be at *after* observing the final concept, predicting the next one. So `raw_output` is the set of concepts whose token appearing next would be a *graph-correct* continuation.

In [3]:
# Sanity check: every consecutive pair in the walk is an edge in the graph.
walk = trace["walk_sequence"]
edges_ok = all(b in graph.adjacency[a] for a, b in zip(walk, walk[1:]))
print(f"Walk length: {len(walk)}")
print(f"Every step is a valid edge: {edges_ok}")
print(f"Final node:               {walk[-1]} ({cfg.concepts[walk[-1]]})")
print(f"Final node coordinates:   {trace['node_coordinates']}")
print(f"Final node neighbors:     {[cfg.concepts[n] for n in graph.adjacency[walk[-1]]]}")

Walk length: 32
Every step is a valid edge: True
Final node:               2 (hint)
Final node coordinates:   (2.0943951023931953,)
Final node neighbors:     ['lot', 'film']


## 2. Templates & token positions

Only one position is exposed: `last`, the trailing separator. That's the position interventions and probes target — it's where the model has consumed the entire walk and is predicting the next concept.

In [4]:
from causalab.neural.pipeline import LMPipeline

pipeline = LMPipeline("gpt2", max_new_tokens=1, max_length=512)
positions = create_token_positions(pipeline)
list(positions.keys())

`torch_dtype` is deprecated! Use `dtype` instead!


['last']

In [5]:
ids = pipeline.load([trace])["input_ids"][0].tolist()
decoded = [pipeline.tokenizer.decode([t]) for t in ids]
pad_id = pipeline.tokenizer.pad_token_id
non_pad = [i for i, t in enumerate(ids) if t != pad_id]

last_idx = positions["last"].index(trace)[0]
print(f"last position resolves to token index {last_idx}: {decoded[last_idx]!r}")
print()
print("Last 6 non-pad tokens:")
for i in non_pad[-6:]:
    marker = "  <-- last" if i == last_idx else ""
    print(f"  idx {i:>4}  {decoded[i]!r}{marker}")

last position resolves to token index 511: '|'

Last 6 non-pad tokens:
  idx  506  '|'
  idx  507  'film'
  idx  508  '|'
  idx  509  'h'
  idx  510  'int'
  idx  511  '|'  <-- last


## 3. Counterfactual generators

`generate_dataset(model, n, seed)` cycles through `node_coordinates` so coverage is uniform across nodes. Each base coordinate is paired with a random distinct counterfactual coordinate; both produce fresh random walks ending at their respective nodes.

In [6]:
for i, ex in enumerate(generate_dataset(model, n_examples=4, seed=0)):
    base = ex["input"]
    cf = ex["counterfactual_inputs"][0]
    base_node = base["walk_sequence"][-1]
    cf_node = cf["walk_sequence"][-1]
    print(f"--- pair {i} ---")
    print(f"  base : coord={base['node_coordinates']}  ends at node {base_node} ({cfg.concepts[base_node]})  "
          f"valid next = {base['raw_output']}")
    print(f"  cf   : coord={cf['node_coordinates']}  ends at node {cf_node} ({cfg.concepts[cf_node]})  "
          f"valid next = {cf['raw_output']}")
    print()

--- pair 0 ---
  base : coord=(0.0,)  ends at node 0 (time)  valid next = ['den', 'lot']
  cf   : coord=(4.1887902047863905,)  ends at node 4 (mind)  valid next = ['film', 'den']

--- pair 1 ---
  base : coord=(1.0471975511965976,)  ends at node 1 (lot)  valid next = ['time', 'hint']
  cf   : coord=(4.1887902047863905,)  ends at node 4 (mind)  valid next = ['film', 'den']

--- pair 2 ---
  base : coord=(2.0943951023931953,)  ends at node 2 (hint)  valid next = ['lot', 'film']
  cf   : coord=(0.0,)  ends at node 0 (time)  valid next = ['den', 'lot']

--- pair 3 ---
  base : coord=(3.141592653589793,)  ends at node 3 (film)  valid next = ['hint', 'mind']
  cf   : coord=(2.0943951023931953,)  ends at node 2 (hint)  valid next = ['lot', 'film']



## Next steps

Run on a real graph + model:

```bash
./scripts/run_exp.sh grid_5x5_8b
./scripts/run_exp.sh cylinder_9x9_8b
```

Outputs land under `artifacts/graph_walk/<model>/<analysis>/...`.